In [2]:
!mkdir -p /kaggle/working/code
!mkdir -p /kaggle/working/CROMA

In [ ]:
# Clones the repository to get the model definitions
!git clone https://github.com/waterdisappear/SARATR-X.git
import sys
sys.path.append('/kaggle/working/SARATR-X/pre-training')

Cloning into 'SARATR-X'...
remote: Enumerating objects: 2316, done.
remote: Counting objects: 100% (2316/2316), done.
remote: Compressing objects: 100% (1639/1639), done.
remote: Total 2316 (delta 694), reused 2220 (delta 636), pack-reused 0 (from 0)
Receiving objects: 100% (2316/2316), 37.54 MiB | 25.77 MiB/s, done.
Resolving deltas: 100% (694/694), done.


In [4]:
!cp /kaggle/input/datasets/anamikapatel8/croma-base/CROMA_base.pt /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/pretrain_croma.py /kaggle/working/CROMA/
!cp /kaggle/input/datasets/anamikapatel8/croma-base/use_croma.py /kaggle/working/CROMA/

In [5]:
!pip install rasterio
!pip install timm==0.5.4 huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 9.9 MB/s eta 0:00:00:00:01
  Attempting uninstall: timm
    Found existing installation: timm 1.0.25
    Uninstalling timm-1.0.25:
      Successfully uninstalled timm-1.0.25


In [ ]:
import os
import json

# Confirm these paths match Data path
S1_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
S2_ROOT = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"

def get_chip_id(folder_name):
    # Extracts the coordinate suffix
    return folder_name.split("_")[-1]

def build_file_list():
    if not os.path.exists(S1_ROOT) or not os.path.exists(S2_ROOT):
        print(f"ERROR: Root paths not found. Please verify S1_ROOT and S2_ROOT.")
        return []

    s1_folders = sorted(os.listdir(S1_ROOT))
    s2_folders = sorted(os.listdir(S2_ROOT))
    s2_dict = {get_chip_id(f): f for f in s2_folders}
    
    pairs = []
    
    # Required files to form a valid paired chip
    s1_required = ["VV.tif", "VH.tif", "LabelWater.tif"]
    # CROMA expects 12 bands 
    s2_required = [f"B{i}.tif" for i in range(1, 10)] + ["B8A.tif", "B11.tif", "B12.tif"]

    print("--- Starting File Integrity Audit ---")

    for s1_f in s1_folders:
        cid = get_chip_id(s1_f)
        if cid in s2_dict:
            s1_path = os.path.join(S1_ROOT, s1_f)
            s2_path = os.path.join(S2_ROOT, s2_dict[cid])
            
            # Verifying SAR files exist for this chip
            s1_valid = all(os.path.exists(os.path.join(s1_path, f)) for f in s1_required)
            
            # Verifying multi-spectral files exist for this chip
            s2_valid = all(os.path.exists(os.path.join(s2_path, f)) for f in s2_required)
            
            if s1_valid and s2_valid:
                pairs.append((s1_f, s2_dict[cid]))

    print(f"Total S1 folders: {len(s1_folders)}")
    print(f"Total S2 folders: {len(s2_folders)}")
    print(f"Successfully matched and verified pairs: {len(pairs)}")
    
    if len(pairs) > 0:
        print(f"\nExample Matched Pair:")
        print(f" S1: {pairs[100][0]}")
        print(f" S2: {pairs[100][1]}")
    
    return pairs

matched_pairs = build_file_list()

# Save to a JSON file 
with open("/kaggle/working/matched_pairs.json", "w") as f:
    json.dump(matched_pairs, f)

print(f"Successfully saved {len(matched_pairs)} pairs to matched_pairs.json")

--- Starting File Integrity Audit ---
Total S1 folders: 900
Total S2 folders: 900
Successfully matched and verified pairs: 900

Example Matched Pair:
 S1: S1A_IW_GRDH_1SDV_20180507T160424_20180507T160449_021800_025A09_981B_01536-10752
 S2: 20180507T074611_20180507T080728_T36NXF_01536-10752
Successfully saved 900 pairs to matched_pairs.json


In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

checkpoint_path = hf_hub_download(
    repo_id="waterdisappear/SARATR-X",
    filename="mae_hivit_base_1600ep.pth",
    subfolder="pre-training",  # This was the missing link!
    token=hf_token
)

print(f"Success! Weights are at: {checkpoint_path}")

pre-training/mae_hivit_base_1600ep.pth:   0%|          | 0.00/263M [00:00<?, ?B/s]

Success! Weights are at: /root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth


In [ ]:
%%writefile /kaggle/working/code/dataset_can_use_for_eo.py 
import sys
sys.path.append("/kaggle/working/code")
import os
import numpy as np
import torch
from torch.utils.data import Dataset
import rasterio

class C2SMSDataset(Dataset):
    def __init__(self, s1_root, s2_root, matched_pairs, img_size=224):
        self.s1_root = s1_root
        self.s2_root = s2_root
        self.pairs = matched_pairs
        self.img_size = img_size
        # Bands for CROMA multi-spectral branch
        self.eo_bands = ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12']

    def __len__(self):
        return len(self.pairs)

    def _normalize_sar(self, band):
        """
        SARATR-X specific normalization:
        Scale clipped dB (-35 to 0) to [0, 1] range.
        """
        band = np.nan_to_num(band)
        band = np.clip(band, -35, 0)
        return (band + 35) / 35.0

    def _preprocess_eo(self, eo):
        """CROMA specific normalization: Scale 0-10000 to [-1, 1] range."""
        eo = np.nan_to_num(eo, nan=0.0)
        return (np.clip(eo, 0, 10000) / 10000.0 * 2.0 - 1.0).astype(np.float32)

    def __getitem__(self, idx):
        s1_folder, s2_folder = self.pairs[idx]
        s1_p = os.path.join(self.s1_root, s1_folder)
        s2_p = os.path.join(self.s2_root, s2_folder)
        
        # --- 1. SAR Preprocessing ---
        with rasterio.open(os.path.join(s1_p, "VV.tif")) as src:
            vv = self._normalize_sar(src.read(1))
        with rasterio.open(os.path.join(s1_p, "VH.tif")) as src:
            vh = self._normalize_sar(src.read(1))
            
        # Creates 3rd channel (Ratio) for SARATR-X HiViT backbone
        ratio = np.clip(vh - vv, -1, 1)
        sar_stack = np.stack([vv, vh, ratio], axis=0) # Shape: [3, H, W]
        sar_tensor = torch.from_numpy(sar_stack).float()

        # --- 2. Multi-spectral Preprocessing (CROMA logic) ---
        eo_data = []
        for b in self.eo_bands:
            with rasterio.open(os.path.join(s2_p, f"{b}.tif")) as src:
                eo_data.append(self._preprocess_eo(src.read(1)))
        eo_tensor = torch.from_numpy(np.stack(eo_data, axis=0))

        # --- 3. Labels & Masks ---
        with rasterio.open(os.path.join(s1_p, "LabelWater.tif")) as src:
            s1_label = torch.from_numpy((src.read(1) == 1).astype(np.int64))
        with rasterio.open(os.path.join(s2_p, "LabelWater.tif")) as src:
            s2_label = torch.from_numpy((src.read(1) == 1).astype(np.int64))
        with rasterio.open(os.path.join(s2_p, "LabelCloud.tif")) as src:
            cloud = torch.from_numpy((src.read(1) == 1).astype(np.float32))

        # --- 4. Spatial Resizing to 224x224 for Foundation Models ---
        # Required for positional embeddings of HiViT and CROMA ViT
        sar_tensor = torch.nn.functional.interpolate(
            sar_tensor.unsqueeze(0), size=(self.img_size, self.img_size), mode='bilinear'
        ).squeeze(0)
        
        eo_tensor = torch.nn.functional.interpolate(
            eo_tensor.unsqueeze(0), size=(self.img_size, self.img_size), mode='bilinear'
        ).squeeze(0)
        
        s1_label = torch.nn.functional.interpolate(
            s1_label.unsqueeze(0).unsqueeze(0).float(), size=(self.img_size, self.img_size), mode='nearest'
        ).squeeze(0).squeeze(0).long()
        
        s2_label = torch.nn.functional.interpolate(
            s2_label.unsqueeze(0).unsqueeze(0).float(), size=(self.img_size, self.img_size), mode='nearest'
        ).squeeze(0).squeeze(0).long()

        return sar_tensor, eo_tensor, s1_label, s2_label, cloud

Overwriting /kaggle/working/code/dataset_can_use_for_eo.py


In [ ]:
%%writefile /kaggle/working/code/model_for_saratr.py
import sys
import os
import torch
import torch.nn as nn

sys.path.append('/kaggle/working/SARATR-X/pre-training/models')
sys.path.append('/kaggle/working/SARATR-X/pre-training')

try:
    from models_hivit import hivit_base
    print(">>> SARATR-X HiViT modules successfully imported.")
except ImportError as e:
    print(f"Import failed. Error: {e}")
    raise

class SARATRFloodSegmenter(nn.Module):
    def __init__(self, weight_path, img_size=224):
        super().__init__()
        # Initializes the HiViT backbone
        self.backbone = hivit_base(img_size=img_size)
        
        checkpoint = torch.load(weight_path, map_location='cpu')
        state_dict = checkpoint['model'] if 'model' in checkpoint else checkpoint
        
        # Loads pre-trained weights
        msg = self.backbone.load_state_dict(state_dict, strict=False)
        print(f"Backbone loaded with msg: {msg}")

        # Freezes the foundation model
        for param in self.backbone.parameters():
            param.requires_grad = False
            
        self.segmentation_head = nn.Sequential(
            nn.Conv2d(512, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=1)
        )

    def forward(self, x):
        # Extracting features from HiViT
        features = self.backbone.forward_features(x)
        # Reshapes tokens to 2D grid
        x_2d = features.transpose(1, 2).reshape(-1, 512, 14, 14)
        logits = self.segmentation_head(x_2d)
        return torch.nn.functional.interpolate(logits, size=(224, 224), mode='bilinear', align_corners=False)

Writing /kaggle/working/code/model_for_saratr.py


In [11]:
%%writefile /kaggle/working/code/croma_wrapper.py 
import sys
sys.path.append("/kaggle/working")
import torch
from CROMA.use_croma import PretrainedCROMA

def load_croma_encoder(pretrained_path, image_resolution, mode="joint", device="cpu"):
    """
    Loads CROMA backbone based on requested modality.
    mode: 'joint' (both), 'eo' (optical), or 'sar' (SAR)
    """
    modality_map = {"joint": "both", "eo": "optical", "sar": "SAR"}
    
    model = PretrainedCROMA(
        pretrained_path=pretrained_path,
        size="base",
        modality=modality_map[mode],
        image_resolution=image_resolution
    ).to(device)
    
    model.eval()
    for p in model.parameters():
        p.requires_grad = False
        
    return model, model.encoder_dim, model.patch_size

Writing /kaggle/working/code/croma_wrapper.py


In [ ]:
%%writefile /kaggle/working/code/fusion_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class FoundationFusionSegmenter(nn.Module):
    def __init__(self, croma_eo, saratr_sar, hidden_dim=256):
        super().__init__()
        self.croma      = croma_eo
        self.saratr     = saratr_sar
        self.hidden_dim = hidden_dim          

        # Projection layers to aligning 768 (MS) and 512 (SAR)
        self.eo_proj  = nn.Conv2d(768, hidden_dim, kernel_size=1)
        self.sar_proj = nn.Conv2d(512, hidden_dim, kernel_size=1)

        # Gated fusion unit conditioned on cloud cover ratio — input: [sar_vec | eo_vec | cloud_ratio] = 2*hidden_dim+1
        self.fusion_gate = nn.Sequential(
            nn.Linear(hidden_dim * 2 + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
            nn.Sigmoid()
        )

        self.head = nn.Sequential(
            nn.Conv2d(hidden_dim, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 2, kernel_size=1)
        )

    def forward(self, sar_img, eo_img, cloud=None):   
        B = eo_img.shape[0]

        # 1. Feature Extraction (frozen backbones)
        with torch.no_grad():
            e_out = self.croma(optical_images=eo_img)["optical_encodings"]  # [B, L, 768]
            s_out = self.saratr.forward_features(sar_img)                   # [B, L, 512]

        # 2. Reshape tokens -> spatial feature maps
        L_e = e_out.shape[1]
        h_e = w_e = int(math.sqrt(L_e))
        e_feats = e_out.transpose(1, 2).reshape(B, 768, h_e, w_e)

        L_s = s_out.shape[1]
        h_s = w_s = int(math.sqrt(L_s))
        s_feats = s_out.transpose(1, 2).reshape(B, 512, h_s, w_s)

        # 3. Project to shared hidden dimension
        e_feats = self.eo_proj(e_feats)
        s_feats = self.sar_proj(s_feats)

        # 4. Spatial alignment — SAR grid -> EO grid
        if (h_s, w_s) != (h_e, w_e):
            s_feats = F.interpolate(s_feats, size=(h_e, w_e), mode='bilinear', align_corners=False)

        # 5. Cloud-conditioned gated fusion
        #    g ≈ 1  ->  trust SAR  (high cloud cover)
        #    g ≈ 0  ->  trust EO   (clear sky)
        sar_vec = F.adaptive_avg_pool2d(s_feats, 1).view(B, self.hidden_dim)
        eo_vec  = F.adaptive_avg_pool2d(e_feats, 1).view(B, self.hidden_dim)

        if cloud is not None:
            cloud_ratio = F.adaptive_avg_pool2d(cloud.unsqueeze(1).float(), 1).view(B, 1)
        else:
            cloud_ratio = torch.zeros(B, 1, device=sar_img.device)

        combined = torch.cat([sar_vec, eo_vec, cloud_ratio], dim=1)  # [B, 2*hidden_dim+1]
        g        = self.fusion_gate(combined).view(B, 1, 1, 1)
        fused    = g * s_feats + (1 - g) * e_feats

        # 6. Segmentation head + upsample to input resolution
        logits = self.head(fused)
        return F.interpolate(logits, size=eo_img.shape[-2:], mode='bilinear', align_corners=False)

Overwriting /kaggle/working/code/fusion_model.py


In [ ]:
%%writefile /kaggle/working/code/train_fusion.py
import sys
import os
import json
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from torch.utils.data import DataLoader, Subset
import optuna 

# --- Paths Setup ---
sys.path.append("/kaggle/working/code")
sys.path.append('/kaggle/working/SARATR-X/pre-training')
sys.path.append('/kaggle/working/SARATR-X/pre-training/models')

from dataset_can_use_for_eo import C2SMSDataset
from fusion_model import FoundationFusionSegmenter
from model_for_saratr import SARATRFloodSegmenter
from croma_wrapper import load_croma_encoder

# --- Reproducibility ---
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    os.environ['PYTHONHASHSEED'] = str(seed)

# --- Cloud-Aware Loss (CE + Dice) ---
def cloud_aware_loss(logits, s1_label, s2_label, cloud_mask, ce_criterion, dice_weight=0.8):
    # Dynamic target: S2 for clear pixels, S1 for cloudy ones
    target = (s2_label * (1 - cloud_mask.long())) + (s1_label * cloud_mask.long())

    ce = ce_criterion(logits, target)

    probs    = torch.softmax(logits, dim=1)[:, 1]
    target_f = target.float()
    inter    = (probs * target_f).sum(dim=(1, 2))
    union    = probs.sum(dim=(1, 2)) + target_f.sum(dim=(1, 2))
    dice     = (1 - (2 * inter + 1e-6) / (union + 1e-6)).mean()

    return (1 - dice_weight) * ce + dice_weight * dice

# --- Configuration ---
DEFAULT_DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_CROMA_CKPT   = "/kaggle/working/CROMA/CROMA_base.pt"
DEFAULT_SARATR_CKPT  = "/root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth"
DEFAULT_PAIRS_JSON   = "/kaggle/working/matched_pairs.json"
DEFAULT_S1_ROOT      = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
DEFAULT_S2_ROOT      = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"
DEFAULT_SAVE_PATH_MODEL = "/kaggle/working/fusion_flood_best.pt"
DEFAULT_SAVE_PATH_CKPT  = "/kaggle/working/fusion_flood_checkpoint.pth"
DEFAULT_RESUME_PATH     = "" # Set to "" for fresh training by default
DEFAULT_TOTAL_EPOCHS    = 50

def train_model(trial=None):
    set_seed(42)

    # Hyperparameters to tune 
    if trial:
        batch_size = trial.suggest_categorical("batch_size", [2, 4, 8])
        lr_head = trial.suggest_float("lr_head", 1e-5, 1e-4, log=True)
        lr_fusion_gate_proj = trial.suggest_float("lr_fusion_gate_proj", 5e-5, 5e-4, log=True)
        ce_weight_flood = trial.suggest_float("ce_weight_flood", 5.0, 20.0, step=2.5)
        dice_loss_weight = trial.suggest_float("dice_loss_weight", 0.5, 0.9, step=0.1)
        weight_decay = trial.suggest_float("weight_decay", 1e-3, 5e-2, log=True)
        n_epochs_for_trial = trial.suggest_int("n_epochs_for_trial", 5, 15) # Shorter trials for tuning
    else:
        # Default values for standalone run 
        batch_size = 8
        lr_head = 7.04913389041416e-05
        lr_fusion_gate_proj = 0.00044321344419884115
        ce_weight_flood = 12.5
        dice_loss_weight = 0.7
        weight_decay = 0.004522587542050758
        n_epochs_for_trial = DEFAULT_TOTAL_EPOCHS

    DEVICE = DEFAULT_DEVICE
    CROMA_CKPT = DEFAULT_CROMA_CKPT
    SARATR_CKPT = DEFAULT_SARATR_CKPT
    PAIRS_JSON = DEFAULT_PAIRS_JSON
    S1_ROOT = DEFAULT_S1_ROOT
    S2_ROOT = DEFAULT_S2_ROOT
    SAVE_PATH_MODEL = DEFAULT_SAVE_PATH_MODEL
    SAVE_PATH_CKPT = DEFAULT_SAVE_PATH_CKPT
    RESUME_PATH = DEFAULT_RESUME_PATH

    # --- Data Preparation ---
    with open(PAIRS_JSON, "r") as f:
        matched_pairs = json.load(f)

    full_dataset = C2SMSDataset(S1_ROOT, S2_ROOT, matched_pairs)
    indices      = torch.randperm(len(full_dataset)).tolist()
    split        = int(0.8 * len(full_dataset))

    train_loader = DataLoader(Subset(full_dataset, indices[:split]),  batch_size=batch_size, shuffle=True,  num_workers=2, pin_memory=True)
    val_loader   = DataLoader(Subset(full_dataset, indices[split:]),  batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)

    # --- Model Initialization ---
    print(">>> Initializing FoundationFusionSegmenter (CROMA EO + SARATR-X SAR) for this trial...")
    croma_eo, _, _   = load_croma_encoder(CROMA_CKPT, 224, mode="eo", device=DEVICE)
    saratr_wrapper   = SARATRFloodSegmenter(SARATR_CKPT).to(DEVICE)
    model            = FoundationFusionSegmenter(croma_eo, saratr_wrapper.backbone).to(DEVICE)

    # --- Optimizer ---
    optimizer = torch.optim.AdamW([
        {'params': list(model.eo_proj.parameters()) +
                   list(model.sar_proj.parameters()) +
                   list(model.fusion_gate.parameters()), 'lr': lr_fusion_gate_proj},
        {'params': model.head.parameters(),             'lr': lr_head}
    ], weight_decay=weight_decay)

    # CE: water class weight
    ce_criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, ce_weight_flood]).to(DEVICE))

    # Scheduler: warmup 5 epochs -> flat -> decay to 20% after epoch 40
    # For shorter trials, the decay phase might not be reached or might need adjustment.
    def get_lr_multiplier(epoch):
        if epoch < 5:
            return (epoch + 1) / 5
        elif epoch > 40: 
            return 0.2
        else:
            return 1.0
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=get_lr_multiplier)

    # --- Resume from checkpoint if available ---
    start_epoch = 0
    best_iou    = 0.0

    if not trial and RESUME_PATH and os.path.exists(RESUME_PATH):
        print(f">>> Resuming from checkpoint: {RESUME_PATH}")
        checkpoint  = torch.load(RESUME_PATH, map_location=DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        start_epoch = checkpoint['epoch'] + 1
        best_iou    = checkpoint.get('best_iou', 0.0)
        print(f">>> Resuming from Epoch {start_epoch} | Best IoU so far: {best_iou:.4f}")
    elif not trial:
        print(">>> No checkpoint found. Starting from scratch.")
    else:
        print(f">>> Starting Optuna trial {trial.number} from scratch.")

    # --- Training Loop ---
    for epoch in range(start_epoch, n_epochs_for_trial):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d} [CROMA+SARATR Fusion]")

        for sar, eo, s1_l, s2_l, cloud in pbar:
            sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE, non_blocking=True) for x in [sar, eo, s1_l, s2_l, cloud]]

            # Resize all inputs to 224x224 as per backbone requirement
            sar   = F.interpolate(sar,   size=(224, 224), mode='bilinear',  align_corners=False)
            eo    = F.interpolate(eo,    size=(224, 224), mode='bilinear',  align_corners=False)
            s1_l  = F.interpolate(s1_l.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1).long()
            s2_l  = F.interpolate(s2_l.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1).long()
            cloud = F.interpolate(cloud.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1)

            optimizer.zero_grad()

            logits = model(sar, eo, cloud)

            loss = cloud_aware_loss(logits, s1_l, s2_l, cloud, ce_criterion, dice_loss_weight)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                list(model.eo_proj.parameters()) + list(model.sar_proj.parameters()) +
                list(model.fusion_gate.parameters()) + list(model.head.parameters()),
                max_norm=1.0
            )

            optimizer.step()
            epoch_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}", "lr": f"{optimizer.param_groups[0]['lr']:.2e}"})

        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']

        # --- Validation ---
        model.eval()
        val_ious = []

        with torch.no_grad():
            for sar, eo, s1_l, s2_l, cloud in val_loader:
                sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE, non_blocking=True) for x in [sar, eo, s1_l, s2_l, cloud]]

                sar   = F.interpolate(sar,  size=(224, 224), mode='bilinear',  align_corners=False)
                eo    = F.interpolate(eo,   size=(224, 224), mode='bilinear',  align_corners=False)
                s1_l  = F.interpolate(s1_l.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1).long()
                s2_l  = F.interpolate(s2_l.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1).long()
                cloud = F.interpolate(cloud.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1)

                logits = model(sar, eo, cloud)

                target = (s2_l * (1 - cloud.long())) + (s1_l * cloud.long())

                # Threshold 0.4: tuned for fused confidence (from NB1)
                preds = (torch.softmax(logits, dim=1)[:, 1] > 0.4).long()

                intersection = (preds & target).sum().float()
                union        = (preds | target).sum().float()
                val_ious.append((intersection + 1e-6) / (union + 1e-6))

        avg_iou  = torch.mean(torch.stack(val_ious)).item()

        print(f"Epoch {epoch:02d} | LR: {current_lr:.2e} | Loss: {epoch_loss/len(train_loader):.4f} | Val IoU: {avg_iou:.4f}")

        # Optuna pruning: if current IoU is not good enough, stop trial early
        if trial:
            trial.report(avg_iou, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        # Checkpointing and best model saving only if not a trial
        if not trial:
            if avg_iou > best_iou:
                best_iou = avg_iou
                torch.save(model.state_dict(), SAVE_PATH_MODEL)
                torch.save({
                    'epoch':                epoch,
                    'model_state_dict':     model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'scheduler_state_dict': scheduler.state_dict(),
                    'best_iou':             best_iou,
                }, SAVE_PATH_CKPT)
                print(f">>> Saved Best Model + Checkpoint (IoU: {best_iou:.4f})")

    return avg_iou

# This block will run only if the script is executed directly (not imported)
if __name__ == '__main__':
    print("Running train_fusion.py in standalone mode (no hyperparameter tuning).")
    # To run a full training without Optuna, calling train_model() without arguments
    final_iou = train_model()
    print(f"Standalone training finished. Final IoU: {final_iou:.4f}")

Writing /kaggle/working/code/train_fusion.py


In [ ]:
!python /kaggle/working/code/train_fusion.py

In [ ]:
%%writefile /kaggle/working/code/evaluate_flood.py
import torch
import numpy as np
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from tqdm import tqdm
import os
import json

# Ensures the necessary paths are in sys.path
import sys
sys.path.append("/kaggle/working/code")
sys.path.append('/kaggle/working/SARATR-X/pre-training')
sys.path.append('/kaggle/working/SARATR-X/pre-training/models')

from dataset_can_use_for_eo import C2SMSDataset
from fusion_model import FoundationFusionSegmenter
from model_for_saratr import SARATRFloodSegmenter
from croma_wrapper import load_croma_encoder

# --- Configuration ---
DEFAULT_DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
DEFAULT_CROMA_CKPT   = "/kaggle/working/CROMA/CROMA_base.pt"
DEFAULT_SARATR_CKPT  = "/root/.cache/huggingface/hub/models--waterdisappear--SARATR-X/snapshots/32cc9ecca6c33832579b189221841cc52e4d45e3/pre-training/mae_hivit_base_1600ep.pth"
DEFAULT_PAIRS_JSON   = "/kaggle/working/matched_pairs.json"
DEFAULT_S1_ROOT      = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s1"
DEFAULT_S2_ROOT      = "/kaggle/input/datasets/anamikapatel8/c2sms-floods/c2sms/s2"
DEFAULT_SAVE_PATH_MODEL = "/kaggle/input/datasets/anamikapatel8/croma-saratr-flood-60-94/fusion_flood_best.pt" # Path to the best model weights

DEVICE = DEFAULT_DEVICE
CROMA_CKPT = DEFAULT_CROMA_CKPT
SARATR_CKPT = DEFAULT_SARATR_CKPT
PAIRS_JSON = DEFAULT_PAIRS_JSON
S1_ROOT = DEFAULT_S1_ROOT
S2_ROOT = DEFAULT_S2_ROOT
SAVE_PATH_MODEL = DEFAULT_SAVE_PATH_MODEL

# --- Metrics Calculation ---
def calculate_dice_score(preds, target):
    smooth = 1e-6
    intersection = (preds * target).sum().float()
    dice = (2. * intersection + smooth) / (preds.sum().float() + target.sum().float() + smooth)
    return dice

def calculate_iou(preds, target):
    smooth = 1e-6
    intersection = (preds * target).sum().float()
    union = (preds | target).sum().float() # Element-wise OR to get union of masks
    iou = (intersection + smooth) / (union + smooth)
    return iou

def calculate_precision_recall(preds, target):
    smooth = 1e-6
    true_positives  = (preds * target).sum().float()
    false_positives = (preds * (1 - target)).sum().float()
    false_negatives = ((1 - preds) * target).sum().float()

    precision = (true_positives + smooth) / (true_positives + false_positives + smooth)
    recall    = (true_positives + smooth) / (true_positives + false_negatives + smooth)
    return precision, recall

# --- Data Preparation ---
with open(PAIRS_JSON, "r") as f:
    matched_pairs = json.load(f)

full_dataset = C2SMSDataset(S1_ROOT, S2_ROOT, matched_pairs)
indices      = torch.randperm(len(full_dataset)).tolist()
split        = int(0.8 * len(full_dataset))

# Uses the validation set for evaluation and visualization
val_dataset  = Subset(full_dataset, indices[split:])
# Uses a batch size of 1 for visualization to easily inspect individual images
val_loader   = DataLoader(val_dataset,  batch_size=1, shuffle=False, num_workers=2, pin_memory=True)

# --- Model Initialization and Loading ---
print(">>> Initializing FoundationFusionSegmenter for evaluation...")
croma_eo, _, _   = load_croma_encoder(CROMA_CKPT, 224, mode="eo", device=DEVICE)
saratr_wrapper   = SARATRFloodSegmenter(SARATR_CKPT).to(DEVICE)
model            = FoundationFusionSegmenter(croma_eo, saratr_wrapper.backbone).to(DEVICE)

# Loads the best model's weights
if os.path.exists(SAVE_PATH_MODEL):
    model.load_state_dict(torch.load(SAVE_PATH_MODEL, map_location=DEVICE))
    print(f">>> Loaded model from {SAVE_PATH_MODEL}")
else:
    print(f"Warning: Model weights not found at {SAVE_PATH_MODEL}. Skipping evaluation.")
    sys.exit("Model weights not found. Cannot proceed with evaluation.")

model.eval() # Set model to evaluation mode

all_dice_scores = []
num_visualizations = 10
visualized_count = 0

# Creates directory for saving results
results_dir = "./visualization_results"
os.makedirs(results_dir, exist_ok=True)
print(f"Saving visualizations to: {results_dir}")

print(">>> Evaluating and visualizing...")

# --- Metrics for cloudy scenes ---
cloudy_scene_dice_scores = []
cloudy_scene_iou_scores = []
cloudy_scene_precisions = []
cloudy_scene_recalls = []
CLOUD_COVER_THRESHOLD = 0.20 # 20 percent

with torch.no_grad():
    for i, (sar, eo, s1_l, s2_l, cloud) in enumerate(tqdm(val_loader, desc="Evaluating")):
        sar, eo, s1_l, s2_l, cloud = [x.to(DEVICE, non_blocking=True) for x in [sar, eo, s1_l, s2_l, cloud]]

        # Resize inputs to 224x224 (as done during training)
        sar_resized   = F.interpolate(sar,   size=(224, 224), mode='bilinear',  align_corners=False)
        eo_resized    = F.interpolate(eo,    size=(224, 224), mode='bilinear',  align_corners=False)
        s1_l_resized  = F.interpolate(s1_l.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1).long()
        s2_l_resized  = F.interpolate(s2_l.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1).long()
        cloud_resized = F.interpolate(cloud.unsqueeze(1).float(),  size=(224, 224), mode='nearest').squeeze(1)

        logits = model(sar_resized, eo_resized, cloud_resized)

        # Gets predicted segmentation map (class 1 is flood)
        preds = (torch.softmax(logits, dim=1)[:, 1] > 0.4).long() # Use the same threshold as in training

        # Determines the target based on cloud mask (same logic as cloud_aware_loss)
        target = (s2_l_resized * (1 - cloud_resized.long())) + (s1_l_resized * cloud_resized.long())

        # Calculates Dice score for the current sample
        dice = calculate_dice_score(preds, target)
        all_dice_scores.append(dice.item())

        # Calculates cloud cover percentage
        cloud_pixels = cloud_resized.sum().item()
        total_pixels = cloud_resized.numel()
        cloud_percentage = cloud_pixels / total_pixels

        if cloud_percentage > CLOUD_COVER_THRESHOLD:
            # Calculates metrics for cloudy scenes
            iou_cloudy = calculate_iou(preds, target)
            precision, recall = calculate_precision_recall(preds, target)

            cloudy_scene_dice_scores.append(dice.item())
            cloudy_scene_iou_scores.append(iou_cloudy.item())
            cloudy_scene_precisions.append(precision.item())
            cloudy_scene_recalls.append(recall.item())

        # --- Visualization ---
        if visualized_count < num_visualizations:
            # Calculates IoU for display in title for all visualized samples
            iou = calculate_iou(preds, target)

            fig, axes = plt.subplots(1, 7, figsize=(35, 5)) # Changed from 6 to 7 subplots, adjusted figsize
            fig.suptitle(f"Sample {i} | Dice Score: {dice.item():.4f} | IoU: {iou.item():.4f} | Cloud Cover: {cloud_percentage:.2%}")

            # Defines flood color and alpha for overlay
            flood_color = np.array([0, 0.5, 1.0]) # A brighter blue for flood (RGB)
            flood_alpha = 0.6 # Transparency for flood overlay

            # Defines cloud color and alpha for overlay
            cloud_color = np.array([0.9, 0.9, 0.9]) # Light grey for cloud (RGB)
            cloud_alpha = 0.4 # Transparency for cloud overlay

            # 1. Original SAR (VV band, first channel)
            sar_vv_np = sar.squeeze().cpu().numpy()[0, :, :]
            axes[0].imshow(sar_vv_np, cmap='gray')
            axes[0].set_title('Original SAR (VV)')
            axes[0].axis('off')

            # 2. Original MS (simple RGB from bands B4, B3, B2)
            eo_display = eo.squeeze().cpu().numpy()[:3, :, :]
            # Normalizing for display: scaling to [0, 1]
            eo_min = eo_display.min()
            eo_max = eo_display.max()
            eo_display = (eo_display - eo_min) / (eo_max - eo_min + 1e-6)
            axes[1].imshow(np.transpose(eo_display, (1, 2, 0)))
            axes[1].set_title('Original EO (RGB)')
            axes[1].axis('off')

            # Predicted Flood Mask
            preds_orig_res = F.interpolate(preds.unsqueeze(1).float(), size=eo.shape[-2:], mode='nearest').squeeze().cpu().numpy()
            # Cloud mask at original resolution for blending
            cloud_orig_res = F.interpolate(cloud.unsqueeze(1).float(), size=eo.shape[-2:], mode='nearest').squeeze().cpu().numpy()

            # 3. SAR with Predicted Flood and Cloud Overlay
            sar_rgb_display = np.stack([sar_vv_np, sar_vv_np, sar_vv_np], axis=-1) # Convert grayscale to RGB for overlay
            sar_rgb_display = (sar_rgb_display - sar_rgb_display.min()) / (sar_rgb_display.max() - sar_rgb_display.min() + 1e-6)

            # Creates flood overlay for SAR
            flood_overlay_sar = np.zeros_like(sar_rgb_display)
            flood_overlay_sar[preds_orig_res == 1] = flood_color
            
            # Blends flood onto SAR
            blended_sar = sar_rgb_display * (1 - flood_alpha) + flood_overlay_sar * flood_alpha

            # Creates cloud overlay for SAR
            cloud_overlay_sar = np.zeros_like(sar_rgb_display)
            cloud_overlay_sar[cloud_orig_res == 1] = cloud_color

            # Blends cloud onto the already blended image (flood + sar)
            final_blended_sar = blended_sar * (1 - cloud_alpha) + cloud_overlay_sar * cloud_alpha
            axes[2].imshow(final_blended_sar)
            axes[2].set_title('SAR + Pred. Flood + Cloud')
            axes[2].axis('off')

            # 4. MS with Predicted Flood and Cloud Overlay
            eo_display_transposed = np.transpose(eo_display, (1, 2, 0))

            # Create flood overlay for MS
            flood_overlay_eo = np.zeros_like(eo_display_transposed)
            flood_overlay_eo[preds_orig_res == 1] = flood_color

            # Blend flood onto MS
            blended_eo = eo_display_transposed * (1 - flood_alpha) + flood_overlay_eo * flood_alpha

            # Create cloud overlay for MS
            cloud_overlay_eo = np.zeros_like(eo_display_transposed)
            cloud_overlay_eo[cloud_orig_res == 1] = cloud_color

            # Blends cloud onto the already blended image (flood + MS)
            final_blended_eo = blended_eo * (1 - cloud_alpha) + cloud_overlay_eo * cloud_alpha
            axes[3].imshow(final_blended_eo)
            axes[3].set_title('EO + Pred. Flood + Cloud')
            axes[3].axis('off')
            
            # 5. Cloud Mask (separate)
            axes[4].imshow(cloud_orig_res, cmap='Reds', alpha=0.7) # Using Reds for cloud mask
            axes[4].set_title('Cloud Mask')
            axes[4].axis('off')

            # 6. Flood Ground Truth 
            target_orig_res = F.interpolate(target.unsqueeze(1).float(), size=eo.shape[-2:], mode='nearest').squeeze().cpu().numpy()
            axes[5].imshow(target_orig_res, cmap='Blues', alpha=0.7) # Using Blues for ground truth flood mask
            axes[5].set_title('Ground Truth Flood')
            axes[5].axis('off')

            # 7. Predicted Flood + Cloud Overlay 
            composite_pred_cloud = np.zeros(preds_orig_res.shape + (3,)) # RGB black background
            
            # Adds predicted flood
            composite_pred_flood_only = np.zeros_like(composite_pred_cloud)
            composite_pred_flood_only[preds_orig_res == 1] = flood_color
            composite_pred_cloud = composite_pred_flood_only

            # Overlays cloud mask on top of predicted flood
            cloud_only_overlay = np.zeros_like(composite_pred_cloud)
            cloud_only_overlay[cloud_orig_res == 1] = cloud_color

            # Blends cloud over the predicted flood
            blended_pred_cloud = composite_pred_cloud * (1 - cloud_alpha) + cloud_only_overlay * cloud_alpha
            axes[6].imshow(blended_pred_cloud)
            axes[6].set_title('Predicted Flood + Cloud')
            axes[6].axis('off')


            plt.tight_layout(rect=[0, 0.03, 1, 0.95])
            plt.savefig(os.path.join(results_dir, f'sample_{i}_dice_{dice.item():.4f}_iou_{iou.item():.4f}_cloud_{cloud_percentage:.2%}.png'))
            plt.show()
            plt.close(fig) 
            visualized_count += 1

# Calculates overall average Dice score
if all_dice_scores:
    average_dice = np.mean(all_dice_scores)
    print(f"\nOverall Average Dice Score on Validation Set: {average_dice:.4f}")
else:
    print("No Dice scores were calculated. Check model loading and data processing.")

# Calculates average metrics for cloudy scenes
if cloudy_scene_dice_scores:
    print(f"\n--- Metrics for Scenes with >{CLOUD_COVER_THRESHOLD:.0%} Cloud Cover ---")
    print(f"Average Dice Score (Cloudy Scenes): {np.mean(cloudy_scene_dice_scores):.4f}")
    print(f"Average IoU (Cloudy Scenes): {np.mean(cloudy_scene_iou_scores):.4f}")
    print(f"Average Precision (Cloudy Scenes): {np.mean(cloudy_scene_precisions):.4f}")
    print(f"Average Recall (Cloudy Scenes): {np.mean(cloudy_scene_recalls):.4f}")
else:
    print(f"\nNo scenes with >{CLOUD_COVER_THRESHOLD:.0%} cloud cover found in the validation set.")


Writing /kaggle/working/code/evaluate_flood.py


In [ ]:
!python /kaggle/working/code/evaluate_flood.py

In [32]:
import sys
!{sys.executable} -m pip install optuna

In [ ]:
#finding the optimal hyperparameters for training the croma saratrx fused model
import sys
sys.path.append("/kaggle/working/code")
import optuna
from train_fusion import train_model

# Creating an Optuna study object and optimizing the objective function

study = optuna.create_study(direction="maximize") # Maximize validation IoU
study.optimize(train_model, n_trials=20)

print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))